# Hafta 6 · Kapı, merdiven ve asansör algılama deneyi

Bu defter algılama deneyini baştan sona yürütür. Ağır iş yalnız **Adım 4** (eğitim, GPU). Her adım sonucunu Drive'a yazar; defter kapansa da kaldığı yerden devam edilir.

| Adım | Ne yapar | Süre (tahmini) |
|---|---|---|
| 0 | Drive, kurulum | ~3 dk |
| 1 | Etiketlenecek kareleri seçer, CVAT zip'i hazırlar | ~2 dk |
| — | **Etiketleme (CVAT, elle)** | kare başına ~30–60 sn |
| 2 | CVAT dışa aktarımını YOLO etiketine çevirir | <1 dk |
| 3 | Eğitim / doğrulama / test blok bölmesi | <1 dk |
| 4 | D2–D3: 5 modelin ince ayarı (GPU) | model başına ~5–15 dk |
| 5 | D1–D4: değerlendirme, sonuç tablosu | ~5–10 dk |

**Hazırlık:** Drive'da `MyDrive/Hafta 6 Algılama/betikler/` klasörü oluşturulup `ortak.py, etiket_seti.py, cvat_donustur.py, bol.py, egit.py, degerlendir.py` yüklenir. Kareler Hafta 6 klasöründen okunur (`Hafta 6 Çalışmaları/derslik4_cekim2/frames_1080`, `derslik4_koridor/frames_1080`).

Adım 1–3 GPU istemez; etiketleme bitene kadar CPU çalışma zamanı yeterlidir. Adım 4–5 için **L4 GPU** seçilir.

In [ ]:
# Adım 0 · Drive ve kurulum
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, sys, subprocess
ALG = '/content/drive/MyDrive/Hafta 6 Algılama'
shutil.copytree(f'{ALG}/betikler', '/content/betikler', dirs_exist_ok=True)
os.chdir('/content/betikler'); sys.path.insert(0, '/content/betikler')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout
      if shutil.which('nvidia-smi') else 'GPU yok (Adım 1-3 için sorun değil)')

In [ ]:
%%capture kurulum
!pip install -q "ultralytics==8.4.208" "git+https://github.com/ultralytics/CLIP.git"


In [ ]:
import ultralytics, torch; print('ultralytics', ultralytics.__version__, '· torch', torch.__version__, '· CUDA', torch.cuda.is_available())

## Adım 1 · Etiketlenecek kareler

Sınıftan her 6. kare (~250), koridordan her 3. kare (~141). Ayrı bir test çekimi (başka kat/bina) yapılırsa karelerinin klasörü `--ayri-test` ile eklenir; bu kareler yalnız teste gider.

Çıktılar `Hafta 6 Algılama/etiket/`: `cvat_yukle.zip` (CVAT'a yüklenecek kareler), `cvat_etiketler.json` (CVAT etiket tanımı), `manifest.csv`.

In [ ]:
!python etiket_seti.py --k sinif=6 koridor=3
# Ayrı test çekimi varsa:
# !python etiket_seti.py --k sinif=6 koridor=3 --ayri-test "/content/drive/MyDrive/.../frames_1080" 

## Etiketleme (CVAT, elle)

1. [app.cvat.ai](https://app.cvat.ai) → **Projects → +** → ad: `derslik4_algilama`. **Labels → Raw** sekmesine `etiket/cvat_etiketler.json` içeriği yapıştırılır (3 sınıf ve öznitelikleri gelir).
2. Projede **Tasks → +** → `cvat_yukle.zip` yüklenir (tek görev).
3. Her kare **Etiketleme kılavuzu**na göre **çokgen (polygon)** ile etiketlenir. Nesnesi olmayan kare boş bırakılır; görevdeki her kare etiketlenmiş sayılır.
4. **Actions → Export task dataset → CVAT for images 1.1** (görüntüsüz) → zip, Drive'da `Hafta 6 Algılama/cvat_export.zip` adıyla saklanır.

İki kişi aynı 50 kareyi ayrı etiketleyip farklılıkları konuşursa kurallar netleşir.

In [ ]:
# Adım 2 · CVAT → YOLO
!python cvat_donustur.py "/content/drive/MyDrive/Hafta 6 Algılama/cvat_export.zip"


In [ ]:
# Adım 3 · Blok bölme ve yerel veri seti
!python bol.py --blok 15
!python bol.py --kur

Bölme özeti `veri/bolme.json` içindedir. Bir sınıf doğrulama veya testte hiç yoksa uyarı çıkar; o sınıf için o kümede sonuç verilemez (ör. asansör yalnız koridorun bir bölümünde). Bu durumda ayrı test çekimi gerekir.

In [ ]:
# Adım 4 · Eğitim (L4 GPU). Kesilirse aynı hücre yeniden çalıştırılır, kaldığı yerden sürer.
!python bol.py --kur          # yeni oturumda yerel kopya yoksa yeniden kurar
!python egit.py

In [ ]:
# Adım 5 · Değerlendirme
!python degerlendir.py
from IPython.display import Markdown, display
display(Markdown(open('/content/drive/MyDrive/Hafta 6 Algılama/sonuclar/sonuclar.md').read()))

In [ ]:
# Oturumu kapatmadan önce (Drive'a yazma tamamlansın)
drive.flush_and_unmount()